<a href="https://colab.research.google.com/github/amulyasundaresan/Midterm-1/blob/main/Midterm_1_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np

In [12]:
DECISIONS = []

def log(file_name, step, decision, rows_affected):
    DECISIONS.append({'file':file_name, 'step':step, 'decision': decision, "rows_affected":rows_affected})
    print( f"[{file_name}]{step}"
          f'{decision}({rows_affected} row(s))')
def show_log():
  return pd.DataFrame(DECISIONS)

# Cleaning

For each file, we went through and did some standard cleaning procedures, such as standardizing column names, getting rid of duplicates, converting columns to numeric, etc. Then, using information from analyizing the data, we went in and cleaned any specific problems that the file had. For example, missing values in indexes or fixing coordinates. Each file has a pipeline function that deals with all of these specific issues at once.

### Distribution Center

In [13]:

dist_df = pd.read_csv("https://raw.githubusercontent.com/amulyasundaresan/Midterm-1/refs/heads/main/Data/distribution_centers.csv")

raw = len(dist_df)
old_col = dist_df.columns.tolist()

def clean_distribution_center(dist_df):
  #Standardizing column names
  dist_df.columns = (dist_df.columns.str.strip().str.lower())
  log('dist_df', 'Columns', 'Standardized column names to be lowercase', len(dist_df))

  #Removing duplicate rows
  dist_before = len(dist_df)
  dist_df= dist_df.drop_duplicates()
  log('dist_df', 'Duplicates', 'Removed duplicate rows', dist_before - len(dist_df))

  #Removed all ~ from surge_capacity
  affected = (dist_df['surge_capacity'].astype(str).str.contains('~',na=False).sum())
  dist_df['surge_capacity']=(dist_df['surge_capacity'].astype(str).str.replace('~',"",regex=False))
  log('dist_df','Numeric Formatting', "Removed '~' from surge_capacity", affected)

  #Converting numeric columns into numeric
  num_col = [ 'avg_daily_shipments', 'surge_capacity', 'distance_to_orlando_mi', 'cost_per_shipment']
  for col in num_col:
    dist_df[col]= pd.to_numeric(dist_df[col], errors='coerce')
  log('dist_df', 'Numeric Conversion', 'Converted numeric columns to numeric', len(dist_df))

  #Taking care of missing values in cost_per_shipment
  missing = dist_df['cost_per_shipment'].isna().sum()
  dist_df['cost_per_shipment']=dist_df['cost_per_shipment'].fillna(dist_df['cost_per_shipment'].median())
  log('dist_df', 'Missing Values', 'Filled missing cost_per_shipment with median', missing)

  return dist_df


### Product Catalog

In [14]:
prod_df = pd.read_csv('https://raw.githubusercontent.com/amulyasundaresan/Midterm-1/refs/heads/main/Data/product_catalog.csv')
raw_prod = len(prod_df)

def clean_product_catalog(prod_df):
  #Standardizing column names
  prod_df.columns = (prod_df.columns.str.strip().str.lower())
  log('prod_df', 'Columns', 'Standardized column names to be lowercase', len(prod_df))

  #Removed Duplicates
  prod_before = len(prod_df)
  prod_df = prod_df.drop_duplicates()
  log('prod_df', 'Duplicates', 'Removed duplicate rows', prod_before - len(prod_df))

  #Removing extra spaces
  object_cols = prod_df.select_dtypes(include='object').columns
  for col in object_cols:
    prod_df[col] = prod_df[col].str.strip()
  log('prod_df', 'Extra Spaces', 'Removed extra spaces from object columns', len(prod_df))

  #Standarizing categories
  cat_map_prod = {
      'breakfast foods': 'Breakfast',
      'breakfast': 'Breakfast',
      'emergency': 'Emergency Supplies',
      'emergency supplies': 'Emergency Supplies',
      'grocery': 'Grocery',
      'groceries': 'Grocery',
      'baby': 'Baby Products',
      'baby products': 'Baby Products',
      'alcohol': 'Alcohol',
      'alcohol drinks': 'Alcohol',
      'alcoholic beverages': 'Alcohol',
      'game & entertainment': 'Game & Entertainment',
      'game': 'Game & Entertainment',
      'entertainment': 'Game & Entertainment',
      'health': 'Health',
      'household': 'Household',
      'snacks': "Snacks"
  }

  affect = prod_df['category'].astype(str).str.lower().isin(cat_map_prod.keys()).sum()
  prod_df['category'] = prod_df['category'].astype(str).str.lower().replace(cat_map_prod)
  log('prod_df', 'Categories', 'Standardized categories', affect)

  #Converting unt_cost to numeric
  prod_df['unit_cost'] = pd.to_numeric(prod_df['unit_cost'], errors='coerce')
  log('prod_df', 'Numeric Conversion', 'Converted unt_cost to numeric', len(prod_df))

  #Finding and filling in missing unit_cost values
  #Grouping by category so each median better reflects what the missing value will likely be, instead of overall median
  missing = prod_df['unit_cost'].isna().sum()
  prod_df['unit_cost'] = prod_df.groupby('category')['unit_cost'].transform(lambda x: x.fillna(x.median()))
  log('prod_df', 'Missing Values', 'Filled missing unt_cost with median', missing)

  #Standardizing package_size
  affect_ps = prod_df['package_size'].notna().sum()
  prod_df['package_size']= prod_df['package_size'].astype(str).str.lower().str.replace('-', '', regex=False)
  log('prod_df', 'Package Size', "Standardized packed size format", affect_ps)

  return prod_df


### Walmart Transaction

In [15]:
walmart = pd.read_csv('https://raw.githubusercontent.com/amulyasundaresan/Midterm-1/refs/heads/main/Data/walmart_transactions.csv')
raw_walmart = len(walmart)

def clean_walmart_transactions(walmart):
  #Standardizing column names
  walmart.columns = (walmart.columns.str.strip().str.lower())
  log('walmart', 'Columns', 'Standardized column names to be lowercase', len(walmart))

  #Removing duplicates
  walmart_before = len(walmart)
  walmart = walmart.drop_duplicates().copy()
  log('walmart', 'Duplicates', 'Removed duplicate rows', walmart_before - len(walmart))

  #Standarizing store_id and fix FL710 to be FL-710
  before_id = walmart['store_id'].copy()
  walmart['store_id']= walmart['store_id'].astype(str).str.strip().str.upper().str.replace('_', '', regex=False)
  walmart['store_id']= walmart['store_id'].str.replace(r'^FL(\d+)$', r"FL-\1", regex=True)

  affect_id = (before_id != walmart['store_id']).sum()
  log('walmart', 'Store ID', 'Standardized store_id format', affect_id)

  #Standardizing and converted unit_price to numeric
  affect_unit = walmart['unit_price'].astype(str).str.contains(r"\$", na=False).sum()

  walmart['unit_price'] = walmart['unit_price'].astype(str).str.replace('$', '', regex=False).str.strip()
  walmart['unit_price'] = pd.to_numeric(walmart['unit_price'], errors='coerce')
  log('walmart', 'Numeric Conversion & Format', 'Converted unit_price to numeric', affect_unit)

  #Converting quantity to numeric, using mixed to account for different date formats
  walmart['quantity'] = pd.to_numeric(walmart['quantity'], errors='coerce')
  log('walmart', 'Numeric Conversion', 'Converted quantity to numeric', len(walmart))

  #Identifying and categorizing any return
  returns = (walmart['quantity']<0).sum()
  walmart['is_return'] = (walmart['quantity']<0).astype(int)
  log('walmart', 'Returns', 'Categorized returns', returns)

  # Standardizing categories
  cat_map_walmart = {
      'breakfast foods': 'Breakfast',
      'breakfast': 'Breakfast',
      'emergency': 'Emergency Supplies',
      'emergency supplies': 'Emergency Supplies',
      'grocery': 'Grocery',
      'groceries': 'Grocery',
      'baby': 'Baby Products',
      'baby products': 'Baby Products',
      'game & entertainment': 'Game & Entertainment',
      'game': 'Game & Entertainment',
      'entertainment': 'Game & Entertainment',
  }

  affect_cat = walmart['category'].astype(str).str.lower().isin(cat_map_walmart.keys()).sum()
  walmart['category'] = walmart['category'].astype(str).str.lower().replace(cat_map_walmart)
  log('walmart', 'Categories', 'Standardized categories', affect_cat)

  #Fill in missing values in timestamp
  missing_ts = walmart['timestamp'].isna().sum()
  walmart['timestamp'] = walmart['timestamp'].replace(['NULL', 'N/A', 'nan', 'NaN', ''], pd.NA)
  log('walmart', 'Missing Values', 'Filled missing timestamp with null value', missing_ts)

  #Convert values in timestamp to date time
  walmart['timestamp'] = pd.to_datetime(walmart['timestamp'].astype(str), format='mixed', errors='coerce')

  invalid = walmart['timestamp'].isna().sum()
  log('walmart', 'Date Time Conversion', 'Converted timestamp to date time', len(walmart)-invalid)

  return walmart

### Store Locations

In [16]:
stores_df = pd.read_csv('https://raw.githubusercontent.com/amulyasundaresan/Midterm-1/refs/heads/main/Data/store_locations.csv')
store_raw = len(stores_df)

def clean_store_locations(stores_df):
  #Standardizing column names
  stores_df.columns = (stores_df.columns.str.strip().str.lower())
  log('stores_df', 'Columns', 'Standardized column names to be lowercase', len(stores_df))

  #Getting rid of extra spaces
  object_cols = stores_df.select_dtypes(include='object').columns

  for col in object_cols:
    stores_df[col] = stores_df[col].astype(str).str.strip()
  log('stores_df', 'Extra Spaces', 'Removed extra spaces from object columns', len(stores_df))

  #Standarizing store_id and fixing Fl710
  before_id = stores_df['store_id'].copy()
  stores_df['store_id']= stores_df['store_id'].astype(str).str.strip().str.upper().str.replace('_', '', regex=False)
  stores_df['store_id']= stores_df['store_id'].str.replace(r'^FL(\d+)$', r"FL-\1", regex=True)
  affect_id = (before_id != stores_df['store_id']).sum()
  log('stores_df', 'Store ID', 'Standardized store_id format', affect_id)

  #Standardizing state
  cat_map_state = {
       'FL': 'FL',
       'Florida': 'FL',
       'FLA.': 'FL'
  }
  affect_state = stores_df['state'].astype(str).str.lower().isin(cat_map_state.keys()).sum()
  stores_df['state'] = stores_df['state'].astype(str).str.lower().replace(cat_map_state)
  log('stores_df', 'State', 'Standardized state format', affect_state)

  #Converting latitude &longitude to numeric
  stores_df['latitude'] = pd.to_numeric(stores_df['latitude'], errors='coerce')
  stores_df['longitude'] = pd.to_numeric(stores_df['longitude'], errors='coerce')
  log('stores_df', 'Numeric Conversion', 'Converted latitude to numeric', len(stores_df))

  #Fix missing coordinate
  missing_coors =( stores_df['latitude'].isna()  | stores_df['longitude'].isna()).sum()
  city_coor = {'Tallahassee': (30.4383, -84.2807)}

  for city, (lat, lon) in city_coor.items():
    stores_df.loc[(stores_df['city'] == city) & (stores_df['latitude'].isna()), 'latitude'] = lat
    stores_df.loc[(stores_df['city'] == city) & (stores_df['longitude'].isna()), 'longitude'] = lon
  log('stores_df', 'Missing Values', 'Filled missing coordinates with city average', missing_coors)

  # Remove duplicates of Orlando and Kissimee
  before_sto_loc = len(stores_df)
  stores_df = stores_df.drop_duplicates(subset=['store_id'])
  log('stores_df', 'Duplicates', 'Removed duplicate rows', before_sto_loc - len(stores_df))

  return stores_df

### Inventory and Sales Database

In [18]:
import gdown
import sqlite3

# Google Drive file ID
file_id = '1_RnLo9hSfX0I7M5fuNeVQXBeRT0ZmxWF'
url = f'https://drive.google.com/uc?id={file_id}'

# Downloading file into notebook
gdown.download(url, 'inventory_and_sales.db', quiet=False)

conn = sqlite3.connect('inventory_and_sales.db')

Downloading...
From: https://drive.google.com/uc?id=1_RnLo9hSfX0I7M5fuNeVQXBeRT0ZmxWF
To: /content/database.db
100%|██████████| 352k/352k [00:00<00:00, 62.8MB/s]


In [19]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table'
"""

tables = pd.read_sql(query, conn)
print(tables)

                  name
0  daily_sales_summary
1     inventory_levels


In [20]:
inventory = pd.read_sql('SELECT * FROM inventory_levels', conn)

sales = pd.read_sql('SELECT * FROM daily_sales_summary', conn)

In [21]:
print(pd.read_sql('PRAGMA table_info(inventory_levels)',conn))

pd.read_sql('PRAGMA table_info(daily_sales_summary)',conn)

   cid            name     type  notnull dflt_value  pk
0    0            date     TEXT        0       None   0
1    1        store_id     TEXT        0       None   0
2    2             sku     TEXT        0       None   0
3    3   units_on_hand  INTEGER        0       None   0
4    4  units_on_order  INTEGER        0       None   0
5    5   reorder_point  INTEGER        0       None   0


,cid,name,type,notnull,dflt_value,pk
0,0,date,TEXT,0,None,0
1,1,store_id,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,total_units,REAL,0,None,0
4,4,total_revenue,REAL,0,None,0


#### Inventory

In [22]:
def cleaned_inventory(inventory):
  #standarizing column names
  inventory.columns = inventory.columns.str.strip().str.lower()
  log('inventory', 'Columns', 'Standardized column names to be lowercase', len(inventory))

  #Removing duplicates
  inventory_before = len(inventory)
  inventory = inventory.drop_duplicates()
  log('inventory', 'Duplicates', 'Removed duplicate rows', inventory_before - len(inventory))

  #Standardizing store_id & fixing Fl
  before_inv_store = inventory['store_id'].copy()

  inventory['store_id'] = (inventory['store_id'].astype(str).str.strip().str.upper().str.replace('_', '', regex=False))
  inventory['store_id'] = inventory['store_id'].str.replace(r'^FL(\d+)$', r"FL-\1", regex=True)
  affect_inv_store = (before_inv_store != inventory['store_id']).sum()
  log('inventory', 'Store ID', 'Standardized store_id format', affect_inv_store)

  #Converting date to datetime
  inventory['date'] = pd.to_datetime(inventory['date'], errors='coerce')
  invalid_inv = inventory['date'].isna().sum()
  log('inventory', 'Date Time Conversion', 'Converted date to date time', len(inventory)- invalid_inv)

  #Converting numeric columns to numeric
  num_cols_inv = ['units_on_hand', 'units_on_order', 'reorder_point']

  for col in num_cols_inv:
    inventory[col] = pd.to_numeric(inventory[col], errors='coerce')
  log('inventory', 'Numeric Conversion', 'Converted numeric columns to numeric', len(inventory))

  #Creating a standardized SKU map
  sku_map_inv = {
      'PT_STRAWBERRY': 'POPTART-STRAW',
      'POPTART_STRAWBERRY': 'POPTART-STRAW',
      'PTOWAL-6': 'PAPERTOWEL6',
      "WTR-24": "WTR_24PK",
      "WATER24PK": "WTR_24PK",
      "BAT-AA4": "BATT_AA_4PK",
      "BATT-AA4": "BATT_AA_4PK"
  }
  affect_inv = inventory['sku'].astype(str).str.upper().isin(sku_map_inv.keys()).sum()
  inventory['sku'] = inventory['sku'].astype(str).str.upper().replace(sku_map_inv)
  log('inventory', 'SKU', 'Standardized SKU format', affect_inv)

  #Filling missing values in units_on_order
  missing_inv = inventory['units_on_order'].isna().sum()
  inventory['units_on_order'] = inventory['units_on_order'].fillna(0)
  log('inventory', 'Missing Values', 'Filled missing units_on_order with 0', missing_inv)
  return inventory

####Daily Sales

In [23]:
def clean_daily_sales(sales):
  #Standardizing column names
  sales.columns = sales.columns.str.strip().str.lower()
  log('sales', 'Columns', 'Standardized column names to be lowercase', len(sales))

  #Remove duplicates
  before_sales = len(sales)
  sales = sales.drop_duplicates()
  log('sales', 'Duplicates', 'Removed duplicate rows', before_sales - len(sales))

  #Standardizing store_id
  sales['store_id']= sales['store_id'].astype(str).str.strip().str.upper().str.replace('_','-', regex=False)
  sales['store_id']= sales['store_id'].str.replace(r'^FL(\d+)$', r"FL-\1", regex=True)
  log('sales', 'Store ID', 'Standardized store_id format', len(sales))

  #Converting date to datetime: Using mixed to deal with all date formats
  sales['date'] = pd.to_datetime(sales['date'].astype(str), format='mixed', errors='coerce')
  invalid_sales = sales['date'].isna().sum()
  log('sales', 'Date Time Conversion', 'Converted date to date time', len(sales)- invalid_sales)

  #Standardizing Categories
  cat_map_sales = {
        'breakfast foods': 'Breakfast',
        'breakfast': 'Breakfast',
        'emergency': 'Emergency Supplies',
        'emergency supplies': 'Emergency Supplies',
        'grocery': 'Grocery',
        'groceries': 'Grocery',
        'baby': 'Baby Products',
        'baby products': 'Baby Products',
        'game & entertainment': 'Game & Entertainment',
        'game': 'Game & Entertainment',
        'entertainment': 'Game & Entertainment',
    }

  affect_sales = sales['category'].astype(str).str.lower().isin(cat_map_sales.keys()).sum()
  sales['category'] = sales['category'].astype(str).str.lower().replace(cat_map_sales)
  log('sales', 'Categories', 'Standardized categories', affect_sales)

  #Changing to total_units and total_revenue to numeric
  sales['total_units'] = pd.to_numeric(sales['total_units'], errors='coerce')
  sales['total_revenue'] = pd.to_numeric(sales['total_revenue'], errors='coerce')
  log('sales', 'Numeric Conversion', 'Converted total_units & total_revenue to numeric', len(sales))

  return sales

### Running Pipeline

In [24]:
transactions = clean_walmart_transactions(walmart)
dist_df = clean_distribution_center(dist_df)
prod_df = clean_product_catalog(prod_df)
stores_df = clean_store_locations(stores_df)
inventory = cleaned_inventory(inventory)
sales = clean_daily_sales(sales)

[walmart]ColumnsStandardized column names to be lowercase(25562 row(s))
[walmart]DuplicatesRemoved duplicate rows(30 row(s))
[walmart]Store IDStandardized store_id format(4345 row(s))
[walmart]Numeric Conversion & FormatConverted unit_price to numeric(3026 row(s))
[walmart]Numeric ConversionConverted quantity to numeric(25532 row(s))
[walmart]ReturnsCategorized returns(130 row(s))
[walmart]CategoriesStandardized categories(16540 row(s))
[walmart]Missing ValuesFilled missing timestamp with null value(884 row(s))
[walmart]Date Time ConversionConverted timestamp to date time(24648 row(s))
[dist_df]ColumnsStandardized column names to be lowercase(5 row(s))
[dist_df]DuplicatesRemoved duplicate rows(0 row(s))
[dist_df]Numeric FormattingRemoved '~' from surge_capacity(1 row(s))
[dist_df]Numeric ConversionConverted numeric columns to numeric(5 row(s))
[dist_df]Missing ValuesFilled missing cost_per_shipment with median(1 row(s))
[prod_df]ColumnsStandardized column names to be lowercase(45 row(s

In [25]:
show_log()

,file,step,decision,rows_affected
0,walmart,Columns,Standardized column names to be lowercase,25562
1,walmart,Duplicates,Removed duplicate rows,30
2,walmart,Store ID,Standardized store_id format,4345
3,walmart,Numeric Conversion & Format,Converted unit_price to numeric,3026
4,walmart,Numeric Conversion,Converted quantity to numeric,25532
5,walmart,Returns,Categorized returns,130
6,walmart,Categories,Standardized categories,16540
7,walmart,Missing Values,Filled missing timestamp with null value,884
8,walmart,Date Time Conversion,Converted timestamp to date time,24648
9,dist_df,Columns,Standardized column names to be lowercase,5


###Final Check

In [26]:
print('Walmart Transactions')
print(transactions.shape)
print(transactions.isna().sum())
print(transactions.duplicated().sum())
print(transactions.dtypes)
#The missing 884 timestamps account for about 3% of all data points. I attempted to fix it but I spent to much time on it

Walmart Transactions
(25532, 9)
transaction_id      0
store_id            0
timestamp         884
sku                 0
product_name        0
category            0
quantity            0
unit_price          0
is_return           0
dtype: int64
0
transaction_id            object
store_id                  object
timestamp         datetime64[ns]
sku                       object
product_name              object
category                  object
quantity                   int64
unit_price               float64
is_return                  int64
dtype: object


In [27]:
print('Distribution')
print(dist_df.shape)
print(dist_df.isna().sum())
print(dist_df.duplicated().sum())
print(dist_df.dtypes)


Distribution
(5, 8)
dc_id                     0
dc_name                   0
city                      0
state                     0
avg_daily_shipments       0
surge_capacity            0
distance_to_orlando_mi    0
cost_per_shipment         0
dtype: int64
0
dc_id                      object
dc_name                    object
city                       object
state                      object
avg_daily_shipments         int64
surge_capacity              int64
distance_to_orlando_mi      int64
cost_per_shipment         float64
dtype: object


In [28]:
print('Product Catalog')
print(prod_df.shape)
print(prod_df.isna().sum())
print(prod_df.duplicated().sum())
print(prod_df.dtypes)
#I don't know how to fix the missing categories

Product Catalog
(45, 7)
sku              0
product_name     0
category         0
subcategory     13
unit_cost        0
supplier         0
package_size     0
dtype: int64
0
sku              object
product_name     object
category         object
subcategory      object
unit_cost       float64
supplier         object
package_size     object
dtype: object


In [29]:
print('Store Locations')
print(stores_df.shape)
print(stores_df.isna().sum())
print(stores_df.duplicated().sum())
print(stores_df.dtypes)


Store Locations
(12, 8)
store_id      0
store_name    0
city          0
state         0
zip_code      0
latitude      0
longitude     0
region        0
dtype: int64
0
store_id       object
store_name     object
city           object
state          object
zip_code        int64
latitude      float64
longitude     float64
region         object
dtype: object


In [30]:
print('Inventory')
print(inventory.shape)
print(inventory.isna().sum())
print(inventory.duplicated().sum())
print(inventory.dtypes)


Inventory
(4320, 6)
date              0
store_id          0
sku               0
units_on_hand     0
units_on_order    0
reorder_point     0
dtype: int64
0
date              datetime64[ns]
store_id                  object
sku                       object
units_on_hand              int64
units_on_order           float64
reorder_point              int64
dtype: object


In [31]:
print('Sales')
print(sales.shape)
print(sales.isna().sum())
print(sales.duplicated().sum())
print(sales.dtypes)


Sales
(3600, 5)
date             0
store_id         0
category         0
total_units      0
total_revenue    0
dtype: int64
0
date             datetime64[ns]
store_id                 object
category                 object
total_units             float64
total_revenue           float64
dtype: object
